# Agentic AI: step by step

**Part A: Setup.** Run these cells first in every new Colab session. Colab forgets everything when a session ends, so we rebuild the setup each time.

### Cell 1: Check the GPU
Confirms Colab gave us a GPU. We want `Tesla T4`.

In [1]:
# Cell 1: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### Cell 2: Install helper tools
`zstd` unpacks Ollama's files. `pciutils` and `lshw` let the installer find the GPU. We install them first so the GPU is detected on the first try.

In [2]:
# Cell 2: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

### Cell 3: Install Ollama
Installs the Ollama program. Look for "NVIDIA GPU installed" near the end. The "systemd is not running" warning is normal in Colab.

In [3]:
# Cell 3: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> NVIDIA GPU installed.
>>> Install complete. Run "ollama" from the command line.


### Cell 4: Start the server
The server is the program that waits for requests. It must be running before Python can talk to the model. We run it in the background and save its messages to `ollama.log`.

In [4]:
# Cell 4: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

### Cell 5: Check the server found the GPU
Searches the server log for GPU lines. We want `Tesla T4` and about 14 GiB available.

In [5]:
# Cell 5: Check the server log for GPU detection
!grep -i -E "gpu|cuda|nvidia" ollama.log | head -n 10

time=2026-10-17T13:14:00.391Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"


### Cell 6: Install the Python client
The `ollama` package lets Python send messages to the server.

In [6]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

### Cell 7: Download the model
Downloads Gemma 4 (small size) onto Colab's disk. We use Python so the output stays clean. A reply of `success` means it worked.

In [7]:
# Cell 7: Download the model through the Python client
import ollama

print(ollama.pull("gemma4:e2b").status)

success


### Cell 8: First message
A quick test that the whole setup works. The first call is slow because the model loads onto the GPU.

In [8]:
# Cell 8: Send one message and print the reply
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(response.message.content)

Hello, I hope you are having a wonderful day!


## Part B: Stage 1, talking to the model from Python

The call stays the same. Only the message text changes. We try four different messages and compare.

### Cell 9: Ask a plain question
The simplest use. We send one question and print the reply.

In [10]:
# Cell 9: Ask a plain question
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is a Bill of Materials (BOM) in two sentences?"}],
)

print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list of all the raw materials, sub-assemblies, parts, and components required to manufacture a finished product. It is essential for tracking inventory, calculating production costs, managing procurement, and ensuring the efficient assembly of the final item.


### Cell 10: Control the length
The model follows instructions about size. We ask for exactly one sentence, then compare with Cell 9.

In [11]:
# Cell 10: Ask for a very short answer

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "In one short sentence, what is a Bill of Materials (BOM)?"}],)
print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list of all the raw materials, components, and parts required to manufacture a finished product.


### Cell 11: Control the style
We explain the same idea to a different reader. The facts stay the same, and the wording changes.

In [12]:
# Cell 11: Explain the same idea to a child

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Explain what a Bill of Materials (BOM) is to a child."}],)
print(response.message.content)

Imagine you want to build the coolest LEGO spaceship ever!

A **Bill of Materials** (we can call it the "BOM" for short) is like a super important **shopping list** or a **secret instruction sheet** that tells you *exactly* everything you need to build something.

### Here is how it works:

**1. It's a List:**
The BOM is just a long list of every single piece of stuff needed.

*   For your LEGO spaceship, the BOM would say:
    *   3 big blue bricks
    *   1 red wing piece
    *   4 tiny wheels
    *   1 piece of sparkly silver for the antenna

**2. It Stops You From Forgetting Things:**
If you just started building without a list, you might accidentally forget the tiny wheels! The BOM makes sure you have every single piece before you start. It keeps you organized!

**3. It Tells You Exactly How Much:**
Sometimes the list doesn't just say "bricks"—it says "3 big blue bricks." It tells you the exact count so you don't run out halfway through!

### Think of it like this:

*   **If you a

### Cell 12: Control the format
We ask for a table. Later, the same idea will let us ask for structured data that code can read.

In [13]:
# Cell 12: Ask for a markdown table
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Make a markdown table of 3 parts in a simple desk lamp: part name, material, and quantity."}],
)
print(response.message.content)

| Part Name | Material | Quantity |
| :--- | :--- | :--- |
| Lamp Base | Metal or Plastic | 1 |
| Lamp Arm/Neck | Metal or Plastic | 1 |
| Lampshade | Fabric or Plastic | 1 |


### Cell 13: Improve a message
Earlier, the model skipped a pronunciation. A clearer instruction gives a more complete answer.

In [14]:
# Cell 13: Ask for hello in four languages with a pronunciation for every word

response = ollama.chat(
    model="gemma4:e2b",
    messages = [{"role": "user", "content": (
        "Say hello in Georgian, Farsi, Welsh and Japanese. "
        "Use one greeting per language. "
        "Give the pronunciation of every greeting in parentheses."
    )}]
)
print(response.message.content)


**Georgian:** გამარჯობა (ga-mar-jo-ba)

**Farsi:** سلام (salām)

**Welsh:** Helô (hel-oh)

**Japanese:** こんにちは (konnichiwa)


## Stage 1 result: the message controls the answer

| Cell | What we changed | Result |
|---|---|---|
| 9 | A plain question | A normal answer |
| 10 | Asked for one short sentence | A shorter answer |
| 11 | Asked for a child's level | Simpler wording |
| 12 | Asked for a table | A table |
| 13 | Added clearer rules | A more complete answer |

**Lesson:** the model does what the message says. Clear instructions give better answers.

## Part C: Stage 2, the model has no memory

Each call to `ollama.chat` starts fresh. We test this with two separate calls.

### Cell 15: Call one, tell the model a fact

In [15]:
# Cell 15: Call one - tell the model a fact
first = ollama.chat(
    model = "gemma4:e2b",
    messages = [{"role": "user", "content": "My favorite color is green."}]
)

print(first.message.content)

Green is a wonderful color! It's often associated with nature, growth, freshness, and tranquility.

What is it about the color green that you like the most?


### Cell 16: Call two, ask about the fact
This is a new call. We send only the new question.

In [16]:
# Cell 16: Second call - ask about the fact from the first call
second = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is my favourite colour?"}],
)
print(second.message.content)

I don't have access to your personal information, so I don't know what your favorite color is.

You'll have to tell me! 😊


In [17]:
# Cell 17: Inspect what goes in and what comes out
sent = [{"role": "user", "content": "Say hi in three words."}]

response = ollama.chat(model="gemma4:e2b", messages=sent)

print(type(sent))              # what we sent: a list
print(len(sent))               # how many messages in the list
print(type(response.message))  # what came back: one Message object
print(response.message.role)   # who wrote it
print(response.message.content)  # the text

<class 'list'>
1
<class 'ollama._types.Message'>
assistant
Hi there!


## Messages and roles

| Term | Meaning |
|---|---|
| `messages` | The list we send in. It holds the whole conversation. |
| `response.message` | One message coming back: the model's reply. |
| `role` | Who wrote the message. |
| `content` | The text of the message. |

| Role | Written by | Purpose |
|---|---|---|
| `system` | Developer | Standing rules for the whole chat |
| `user` | Person | Questions and instructions |
| `assistant` | Model | Its earlier replies |
| `tool` | Our code | Results of tool calls (explained in Stage 5) |

In [18]:
# Cell 18: Use a system message to set the rules
reply = ollama.chat(
    model="gemma4:e2b",
    messages=[
        {"role": "system", "content": "You are a strict assistant. Answer in 5 words or fewer."},
        {"role": "user", "content": "What is a Bill of Materials?"},
    ],
)
print(reply.message.content)

List of required parts for a product.


In [19]:
# Cell 19: Count the words in the reply
print(len(reply.message.content.split()))

7


### Cell 20: Send the full conversation
We put all three messages in one list: what we said, what the model replied, and the new question.

In [20]:
# Cell 20: Send the full conversation, not just the latest question
history = [
    {"role": "user", "content": "My favourite colour is green."},
    {"role": "assistant", "content": first.message.content},  # the model's reply from Cell 15
    {"role": "user", "content": "What is my favourite colour?"},
]

reply = ollama.chat(model="gemma4:e2b", messages=history)
print(reply.message.content)

Based on what you told me earlier, your favourite colour is **green**.


### Cell 21: Plant a false reply
We change what the assistant "said" and see which one the model believes.

In [21]:
# Cell 21: Replace the model's earlier reply with a made up one

# A list of dictionaries representing the conversation history
history = [ 
    {"role":"user", "content": "My favourite colour is green."},
    {"role":"assistant", "content": "Your favourite colour is blue."},  # made up reply
    {"role":"user", "content": "What is my favourite colour?"}
]

reply = ollama.chat(
    model="gemma4:e2b",
    messages=history
)

print(reply.message.content)

Your favourite colour is green.


## Part C result: the history is the only memory

| Cell | History sent | Model answer |
|---|---|---|
| 16 | None (new call) | Does not know the colour |
| 20 | Real conversation | Green |
| 21 | Planted assistant reply saying red | Green (the user's own words won) |

**Lesson:** the model answers from the history it is given. In this test it trusted the user's words over a planted assistant reply, but one test is not proof. The code that builds the history must be trusted.

### Cell 22: Start an empty history
This list will hold the whole conversation.

In [22]:
# Cell 22: Start an empty conversation history

chat_history = []

### Cell 23: The `ask` helper
It adds our message, sends the full history, and stores the reply.

In [23]:
# Cell 23: Helper that sends one message and keeps the full history

def ask(text):
    # Add the user's message to the shared history
    chat_history.append({"role": "user", "content": text})
    # Send the whole history so the model sees earlier turns
    response = ollama.chat(model = "gemma4:e2b", messages = chat_history)
    answer = response.message.content

    #store the models reply so the next call can see it
    chat_history.append({"role": "assistant", "content": answer})
    return answer

### Cell 24: Test the memory
We only type the new question. The helper builds the history.

In [24]:
# Cell 24: Test that the model now remembers across calls
print(ask("My favourite colour is green."))
print("---")
print(ask("What is my favourite colour?"))

Green is a wonderful colour! It's often associated with nature, growth, freshness, and tranquility. 🌿💚
---
Based on what you told me earlier, your favorite color is **green**! 💚


In [25]:
# Cell 25: Print the stored history
for message in chat_history:
    print(message["role"], ":", message["content"][:80])

user : My favourite colour is green.
assistant : Green is a wonderful colour! It's often associated with nature, growth, freshnes
user : What is my favourite colour?
assistant : Based on what you told me earlier, your favorite color is **green**! 💚


### Cell 26: Wipe the history between questions
We empty the notebook after the first question. Does the model still know the colour?

In [26]:
# Cell 26: Empty the history between two questions
chat_history = []                      # start clean
print(ask("My favourite colour is green."))
print("--- wiping the history now ---")
chat_history = []                      # wipe it
print(ask("What is my favourite colour?"))
print(len(chat_history))               # how many messages are stored now?

Green is a wonderful color! It's a very refreshing and calming shade. It reminds many people of nature and growth. 🌿💚
--- wiping the history now ---
I don't have access to your personal information, so I don't know what your favorite color is!

You'll have to tell me! 😊
2


## Part D result: the history grows by itself

| Step | `chat_history` size | What the model sees |
|---|---|---|
| Start | 0 | nothing |
| After question 1 and its reply | 2 | question 1 only (at the moment of the call) |
| After question 2 and its reply | 4 | questions and replies so far |

**Lesson:** the notebook (`chat_history`) is the only memory. Wiping it erases the memory. A longer history means more text to send each time.

## Part D result: wiping the history erases the memory

| Step | `chat_history` size | Model answer |
|---|---|---|
| Question 1 | 2 | Replies to the colour |
| History wiped | 0 | |
| Question 2 | 2 | Does not know the colour |

**Lesson:** the model has no memory of its own. The list we send is all it knows.

## Part E: Stage 4, structured output

Code cannot do much with free text. We ask the model for JSON, which code can read.

Short idea first. Until now, the model gave us free text. A person can read that, but code can’t use it. The quoting project needs answers like “part name, quantity, price” that a program can read and add up. We ask the model for JSON, a plain-text format for organised data, and then Python reads it.

JSON looks like this:

```json
{"part": "circuit breaker", "quantity": 2}
```

### Cell 27: Ask the model for JSON
We describe the shape we want and ask for nothing else.

In [27]:
# Cell 27: Ask for one part as JSON
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "Give me one part of a desk lamp as JSON with these keys: "
        "name (text), material (text), quantity (whole number). "
        "Reply with the JSON only."
    )}],
)
raw = response.message.content
print(raw)
print(type(raw))

```json
{
  "name": "Lampshade",
  "material": "Fabric",
  "quantity": 1
}
```
<class 'str'>


### Cell 28: Turn the text into data
`json.loads` converts JSON text into a Python dictionary. Then we can read one field by its label.

In [28]:
# Cell 28: Parse the JSON text into a Python dictionary
import json

part = json.loads(raw)
print(type(part))
print(part["name"])
print(part["quantity"] + 1)   # proves the quantity is a real number

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

he model wrapped the JSON in three backticks and the word json. That is how chat apps show code blocks, and the model copied the habit. We asked for “the JSON only”, and it still added the wrapper.

How to read the error

Read the last line first:

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Part	Meaning
JSONDecodeError	Python tried to read JSON and failed
Expecting value	It expected the start of a value, like {
line 1 column 1 (char 0)	The problem is at the very first character

The first character is a backtick, not {. So Python gave up straight away. The long list of lines above it only shows the path through the package’s code, and you can skip it.

In [29]:
# Cell 29: Show the first characters of the reply, including hidden ones
print(repr(raw[:12]))

'```json\n{\n  '


### Cell 30: Ask for JSON properly
The `format="json"` option makes the server return clean JSON, so there is no code-block wrapper.

In [31]:
# Cell 30: Ask for JSON with the format option
response = ollama.chat(
    model = "gemma4:e2b",
    messages = [{"role": "user", "content": (
        "Give me one part of a desk lamp as JSON with these keys: "
        "name (text), material (text), quantity (whole number). "
    )}],
    format = "json"
)

raw = response.message.content
print(repr(raw[:12]))
print(raw)


'{\n  "name": '
{
  "name": "Lamp Shade",
  "material": "Fabric",
  "quantity": 1
}


### Cell 31: Parse the clean JSON
Same step as Cell 28, now on clean text.

In [32]:
# Cell 31: Parse the JSON text into a Python dictionary
part = json.loads(raw)
print(type(part))
print(part["name"])
print(part["quantity"] + 1)

<class 'dict'>
Lamp Shade
2


## Part E result: free text vs structured output

| Cell | What we did | Result |
|---|---|---|
| 27 | Asked for "JSON only" in the message | The reply had a code-block wrapper |
| 28 | `json.loads` on that reply | Failed: the first character was a backtick |
| 30 | Added `format="json"` | Clean JSON |
| 31 | `json.loads` on clean JSON | Worked |

**Lesson:** asking nicely in the message is not enough. A format option makes the server enforce the shape.

### Cell 32: What if we do not name the keys?
The reply is valid JSON, but the labels are the model's own choice.

In [33]:
# Cell 32: Ask for JSON without naming the keys
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Give me one part of a desk lamp."}],
    format="json",
)
raw = response.message.content
print(raw)

part = json.loads(raw)
print(list(part.keys()))   # the labels the model chose
print(part["name"])        # does this label exist?

{"part": "The shade"}
['part']


KeyError: 'name'

## Part E lesson: valid JSON is not the same as the right JSON

| What `format="json"` guarantees | What it does not guarantee |
|---|---|
| The reply is valid JSON | The labels are the ones we want |
| No code-block wrapper | The values have the right types |
| | Every required field is present |

## Part E result: no schema, no control (Cell 32)

| Cell | What we did | Result |
|---|---|---|
| 32 | Asked for JSON without naming the keys | `{"part": "The shade"}`, so `part["name"]` raised a KeyError |

**Lesson:** the model chose its own labels. We need to define the shape and check the reply against it.

In [34]:
# Extra example 1: ask for JSON and name the fields in the prompt

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "Give me the name of dynasties in Iran in order and the years they ruled. "
        "name of dynasty(text), start year(whole number), end year(whole number). "
    )}],
    format = "json"
)

raw = response.message.content
print(raw)

{"dynasties": [
    "Achaemenid Empire, 550, 330",
    "Parthian Empire, 247, 224",
    "Sasanian Empire, 224, 651",
    "Abbasid Dynasty, 750, 1258",
    "Seljuk Dynasty, 1037, 1194",
    "Ilkhanate, 1256, 1335",
    "Safavid Dynasty, 1501, 1736",
    "Qajar Dynasty, 1785, 1925",
    "Pahlavi Dynasty, 1925, 1979"
]}


In [35]:
# Extra example 2: ask for JSON with no fields named

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "Give me the name of dynasties in Iran in order and the years they ruled. "
    )}],
    format = "json"
)

raw = response.message.content
print(raw)

ResponseError: prediction aborted, token repeat limit reached (status code: 500)

In [37]:
# Extra example 3: ask for a list of dynasties with named keys, and cap the reply length
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "List 6 dynasties that ruled Iran, in order. "
        "Reply with a JSON object with one key, 'dynasties'. "
        "Its value is a list of objects. Each object has: "
        "'name' (text), 'start_year' (whole number), 'end_year' (whole number). "
        "Use negative numbers for years BCE."
    )}],
    format="json",
    options={"num_predict": 600},   # stop after at most 600 tokens
)
raw = response.message.content
print(raw)

In [38]:
# Extra example 4: diagnose an empty reply
print(repr(response.message.content))      # the answer text
print(response.done_reason)                # why the model stopped
print(len(response.message.thinking or ""))  # how much thinking text was stored

''
length
2111


In [39]:
# Extra example 5: switch thinking off so the answer fits the budget
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "List 6 dynasties that ruled Iran, in order. "
        "Reply with a JSON object with one key, 'dynasties'. "
        "Its value is a list of objects. Each object has: "
        "'name' (text), 'start_year' (whole number), 'end_year' (whole number). "
        "Use negative numbers for years BCE."
    )}],
    format="json",
    think=False,                     # skip the thinking step
    options={"num_predict": 600},    # still cap the reply length
)
print(response.message.content)

{
  "dynasties": [
    {
      "name": "Achaemenid",
      "start_year": -550,
      "end_year": -330
    },
    {
      "name": "Parthian",
      "start_year": -247,
      "end_year": 224
    },
    {
      "name": "Sassanid",
      "start_year": 224,
      "end_year": 651
    },
    {
      "name": "Safavid",
      "start_year": 1501,
      "end_year": 1736
    },
    {
      "name": "Qajar",
      "start_year": 1785,
      "end_year": 1925
    },
    {
      "name": "Pahlavi",
      "start_year": 1925,
      "end_year": 1979
    }
  ]
}


## Part E result: thinking can eat the token budget

| Cell | Setting | Result |
|---|---|---|
| Diagnose | `num_predict=600`, thinking on | Empty answer. `done_reason` was `length`, with 2111 characters of thinking stored. |
| 38 | `think=False` | Clean JSON with the right keys |

**Lesson:** if a reply is empty, check `done_reason` and the thinking text. For simple jobs like filling a form, switch thinking off.

## Part F: define the shape with Pydantic

Short idea first. A schema is a form with fixed boxes: each box has a label and a type of answer. Pydantic is a Python library where you write that form as a small class. It then checks any data against the form and complains if something is wrong.

We learn Pydantic by itself first, with no model. One idea at a time.

Step 1. Install it

### Cell 33: Install Pydantic

In [40]:
# Cell 33: Install Pydantic
!pip install -q pydantic

### Cell 34: Define the form
A `Part` has three boxes. Each box has a label and a type: text, text, whole number.

In [41]:
# Cell 34: Define the shape of one part
from pydantic import BaseModel, ValidationError

class Part(BaseModel):
    name: str        # text
    material: str    # text
    quantity: int    # whole number

### Cell 35: A correct part
We fill every box with the right kind of value.

In [42]:
# Cell 35: Create a valid Part
good = Part(name="Lamp shade", material="Fabric", quantity=2)
print(good)
print(good.name)
print(good.quantity + 1)   # a real number, so maths works

name='Lamp shade' material='Fabric' quantity=2
Lamp shade
3


### Cell 36: A wrong part
We put text in the quantity box. Pydantic should refuse.

In [43]:
# Cell 36: Try to create an invalid Part and read the error
try:
    bad = Part(name="Lamp shade", material="Fabric", quantity="two")
except ValidationError as error:
    print(error)

1 validation error for Part
quantity
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='two', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


### Cell 37: A missing field
We leave out `material`.

In [44]:
# Cell 37: Try to create a Part with a missing field
try:
    bad = Part(name="Lamp shade", quantity=2)
except ValidationError as error:
    print(error)

1 validation error for Part
material
  Field required [type=missing, input_value={'name': 'Lamp shade', 'quantity': 2}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [45]:
# Experiment: a different class name
class Lamp(BaseModel):
    name: str
    material: str
    quantity: int

print(Lamp(name="Base", material="Metal", quantity=1))

name='Base' material='Metal' quantity=1


## Pydantic words

| Word | What it is | Plain meaning |
|---|---|---|
| `pydantic` | A Python package | The toolbox that checks data |
| `BaseModel` | A ready-made class inside Pydantic | A blank form that already knows how to check answers |
| `class` | A Python keyword | "Make a new form" |
| `Part` | A name we chose | Our own form (any name works) |
| `(BaseModel)` | The brackets after the class name | "Built on this ready-made form" |
| `name: str` | A box | A box called `name` that takes text |
| `quantity: int` | A box | A box called `quantity` that takes a whole number |
| `str` | A type | Text |
| `int` | A type | Whole number |
| `Part(...)` | Filling in the form | Makes one filled-in copy, and checks it |
| `good`, `bad` | Variables | Names for the filled-in copies |
| `ValidationError` | An error type from Pydantic | "The form was filled in wrongly" |
| `try` / `except` | Python keywords | "Try this. If it fails, show the message instead of crashing." |


## Part G: make the model fill in the form

Short idea first. Until now, we described the shape in words and hoped. Now we hand the model the form itself. Ollama then forces the reply to follow it, and Pydantic checks it afterwards. This gives two layers: the server enforces the shape, and Pydantic confirms it.

Step 1. Turn the form into a description the server understands

### Cell 38: See the form as a schema
A schema is the form written in a standard format that other programs can read.

In [46]:
# Cell 38: Show the Part form as a JSON schema
print(Part.model_json_schema())

{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'material': {'title': 'Material', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}}, 'required': ['name', 'material', 'quantity'], 'title': 'Part', 'type': 'object'}


### Cell 39: Ask the model to fill in the form
We pass the schema in `format=`. Thinking is off, because this is a simple job.

In [47]:
# Cell 39: Ask for one part, forced into the Part shape
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Give me one part of a desk lamp."}],
    format=Part.model_json_schema(),   # the form, as a schema
    think=False,
)
raw = response.message.content
print(raw)

{
  "name": "Light Bulb",
  "material": "Glass or Plastic",
  "quantity": 1
}


### Cell 40: Check the reply against the form
`model_validate_json` reads the JSON text and checks it against `Part` in one step.

In [48]:
# Cell 40: Parse and validate the reply
part = Part.model_validate_json(raw)
print(part)
print(part.name)
print(part.quantity + 1)

name='Light Bulb' material='Glass or Plastic' quantity=1
Light Bulb
2


## Theory: agent loop vs graph workflow

An agent has to do several steps, not one. There are two main ways to organise those steps: a **loop** and a **graph**.

---

### 1. The agent loop

**Picture:** a person working through a to-do list. They look at the list, do one thing, look at what changed, and decide what to do next. They stop when the list is done.

```
START with a goal
   |
   v
THINK:  what should I do next?   <---------+
   |                                       |
   v                                       |
ACT:    ask for a tool                     |
   |                                       |
   v                                       |
SEE:    read the tool's result  -----------+
   |
   v
Goal done?  yes -> write the final answer
```

**Who does what**

| Who | Job |
|---|---|
| The model | Decides the next step and asks for a tool |
| Our code | Runs the tool and sends the result back |
| The model again | Reads the result and decides whether to continue |

**Example from the quoting project: "Price this switchboard"**

| Round | The model thinks | The model asks for | Result |
|---|---|---|---|
| 1 | I need the parts list | `read_drawing` | 3 breakers, 1 busbar |
| 2 | I need prices | `lookup_price` (breaker) | 12.50 |
| 3 | Same for the busbar | `lookup_price` (busbar) | 80.00 |
| 4 | Add it up | `line_total` | 117.50 |
| 5 | I have everything | no tool | Final answer |

Nobody planned five rounds. The model chose that as it went.

**Strengths and weak points**

| Strength | Weak point |
|---|---|
| Flexible. It copes with questions we did not plan for. | Hard to predict. It may take 3 rounds or 12. |
| Little code to write | It can get stuck repeating itself |
| | Hard to prove it always follows the rules |

**Loop engineering** means making the loop safe and reliable:

| Practice | Why |
|---|---|
| A maximum number of rounds | The loop can never run forever |
| A check on every tool result | A bad result does not spread |
| A clear stop rule | The model knows when it is finished |
| A log of every round | We can see what happened and why |

---

### 2. The graph workflow

**Picture:** a Tube map. It has stations and lines. You can only travel along lines that exist. The map decides which routes are possible.

| Word | Meaning |
|---|---|
| Node | One station: a single job, such as "read drawing" |
| Edge | A line: an arrow saying which node may come next |
| Branch | A fork: "if the check fails, go here, otherwise go there" |
| State | The information carried from station to station |

```
[Read drawing] --> [Extract parts] --> [Check parts list]
                                           |        |
                                      passes        fails
                                           |        |
                                           v        v
                                    [Look up prices] [Ask a human]
                                           |
                                           v
                                    [Build quote] --> END
```

We draw the map in advance. The model works inside each node, but it cannot jump to a station that has no line.

**Strengths and weak points**

| Strength | Weak point |
|---|---|
| Predictable. We know every possible route. | Less flexible. An unplanned question may not fit. |
| Easy to test one node at a time | More design work before anything runs |
| Easy to explain to a customer or an auditor | |

**Graph engineering** means designing the map well:
- what the nodes are
- which arrows connect them
- where the checks and branches go
- where a human steps in
- what the state carries from node to node

---

### 3. Loop vs graph

| | Loop | Graph |
|---|---|---|
| Who decides the next step | The model, as it goes | The designer, in advance |
| Picture | A person with a to-do list | A Tube map |
| Good for | Open questions, exploring | Fixed processes, rules, checks |
| Main risk | Wanders, repeats, surprises | Too rigid for unusual cases |
| Easy to audit | Harder | Easier |

---

### 4. Design idea for the quoting project

The system must be **secure**, **explainable** and suitable for **regulated** sectors. A quote should not depend on the model's mood. So a sensible design is:

> **A graph for the fixed process, with small loops inside single steps.**

- The **graph** sets the fixed steps: read drawing, extract parts, check, price, build quote, human review.
- Inside one node, such as "find the right part", a small **loop** lets the model try a few searches.

This is my suggested design.

---

### 5. Where we use each idea in this course

| Stage | Idea |
|---|---|
| 5 | A simple loop, built by hand |
| 8 | A loop inside agentic RAG |
| 10 | Checks and a graph-style structure |

---

### Quick check

1. In the loop, who decides to stop: the model, our code, or both?
2. If a customer needs proof of exactly which steps produced a quote, would you pick a loop or a graph, and why?

## Part 2: Stage 5 again, from Cell 41

We go back to the start of Stage 5 and take it slowly.

The one idea

A tool is a normal Python function that the model may ask us to run. The model never runs it. It sends a request, our code runs the function, and we hand the result back.

The loop has four steps. We build them one at a time.

## Part H: Stage 5, tools and the agent loop

The model only writes text. A **tool** is a normal Python function that we let the model ask for.
The model never runs it. It sends a request, our code runs the function, and we return the result.

| Step | Who | What |
|---|---|---|
| 1 | Us | Write the function |
| 2 | Us and model | Offer the function, and look at the model's request |
| 3 | Our code | Run the function |
| 4 | Us | Send the result back (role `tool`) so the model can write the final answer |

### Cell 41: Write a tool (plain Python, no model yet)

This is the tool. It adds up one line of a quote. The model has not seen it yet.

| Piece | Meaning |
|---|---|
| `def line_total(...)` | Makes a function called `line_total` |
| `unit_price: float` | The first input is a number that may have decimals |
| `quantity: int` | The second input is a whole number |
| `-> float` | The function returns a number with decimals |
| Text in triple quotes | A **docstring**. The model reads it to decide when to use the tool. |

**Expected output:** `50.0`

In [49]:
# Cell 41: A simple tool that adds up one quote line
def line_total(unit_price: float, quantity: int) -> float:
    """Return the total cost of one quote line: unit price times quantity."""
    return unit_price * quantity

print(line_total(12.5, 4))

50.0


### Cell 42: Offer the tool and read the model's request

We pass the function in `tools=`. The model should **not** answer with a number. It should send a **request** that names the tool and its inputs.

| Piece | Meaning |
|---|---|
| `tools=[line_total]` | The list of tools the model may ask for |
| `think=False` | Skip the thinking step, as this is a simple job |
| `response.message.content` | The text part of the reply. It may be empty. |
| `response.message.tool_calls` | The model's request: which tool, with which inputs |

**What a good result looks like:** `content` is empty, and `tool_calls` names `line_total` with `unit_price=12.5` and `quantity=4`.

In [50]:
# Cell 42: Offer the tool and print the model's tool request
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is the total for 4 circuit breakers at 12.50 each?"}],
    tools=[line_total],
    think=False,
)
print(repr(response.message.content))
print(response.message.tool_calls)

''
[ToolCall(function=Function(name='line_total', arguments={'quantity': 4, 'unit_price': 12.5}))]


## What comes next

Stage 5 (tools and the agent loop) continues in `02-tools-and-agent-loop.ipynb`, which restarts the stage from the first step.
